In [5]:
# ==================================================================================
# 💎 TITAN DERMAL STACK V2.3 – FORCED BRIGHTENER SLOT
# ==================================================================================
# Changes vs v2.2:
#   - For LED Mask & High Frequency Wand, pre‑select the best Brightening active
#     (if available, budget allows, and irritation rules pass), then fill remaining slots.
# ==================================================================================

def is_high_irritant(row):
    return (row["family"] in ["Retinoid","Exfoliant"] or
            (row["family"] == "Brightening" and row["strength"] == "High"))

def build_protocol_constrained_v3(
        device_name,
        max_topicals=4,
        max_supps=2,
        topical_budget=85.0,
        supp_budget=70.0
    ):
    needs = device_needs[device_name]
    sneeds = device_supp_needs[device_name]
    base_price = device_base_price[device_name]

    # ------------------ TOPICAL SELECTION ------------------
    df_t = topical_df.copy()
    df_t["high_irritant"] = df_t.apply(is_high_irritant, axis=1)

    df_t["match_score"] = df_t["family"].apply(lambda f: 3 if f in needs else 0)
    df_t["safety_score"] = df_t["irritation_risk"].apply(lambda r: 4 - (r - 1))
    df_t["value_score"]  = df_t["typical_price"].apply(
        lambda p: 3 if p <= 25 else 2 if p <= 35 else 1
    )
    df_t["total_score"]  = df_t["match_score"] + df_t["safety_score"] + df_t["value_score"]
    df_t = df_t.sort_values("total_score", ascending=False)

    chosen = []
    t_cost = 0.0
    high_irritant_used = False
    used_families = set()

    def violates_combo_constraints(row):
        fam = row["family"]
        if fam == "Exfoliant" and ("Retinoid" in used_families):
            return True
        if fam == "Retinoid" and ("Exfoliant" in used_families):
            return True
        return False

    # ---------- STEP 0: Reserve a brightener slot for LED & HF ----------
    if device_name in ["LED Mask","High Frequency Wand"]:
        br_cand = df_t[df_t["family"] == "Brightening"].sort_values("total_score",
                                                                    ascending=False)
        if not br_cand.empty:
            br = br_cand.iloc[0]
            # only add if budget & irritation rules allow
            if (br["typical_price"] <= topical_budget and
                not (br["high_irritant"] and high_irritant_used)):
                chosen.append(br)
                t_cost += br["typical_price"]
                used_families.add(br["family"])
                if br["high_irritant"]:
                    high_irritant_used = True

    # ---------- STEP 1: Fill remaining slots ----------
    for _, r in df_t.iterrows():
        if len(chosen) >= max_topicals:
            break
        if r["name"] in [c["name"] if isinstance(c, dict) else c.name
                         for c in chosen]:
            continue
        if t_cost + r["typical_price"] > topical_budget:
            continue
        if r["high_irritant"] and high_irritant_used:
            continue
        if violates_combo_constraints(r):
            continue

        chosen.append(r)
        t_cost += r["typical_price"]
        used_families.add(r["family"])
        if r["high_irritant"]:
            high_irritant_used = True

    kit_topicals = pd.DataFrame(chosen).head(max_topicals)

    # ------------------ SUPPLEMENT SELECTION (same as v2.2) ------------------
    df_s = supp_df.copy()

    def supp_bonus(row):
        fam = row["family"]
        if device_name in ["LED Mask","High Frequency Wand","Microcurrent Massager","Electric Gua Sha"]:
            if fam in ["Collagen","HA"]:
                return 2
        if device_name == "Knee Massager":
            if fam in ["Collagen","HA"]:
                return 3
        return 0

    df_s["match_score"] = df_s["family"].apply(lambda f: 3 if f in sneeds else 0)
    df_s["bonus"] = df_s.apply(supp_bonus, axis=1)
    df_s["total_score"] = df_s["match_score"] + df_s["evidence_score"] + df_s["bonus"]
    df_s = df_s.sort_values("total_score", ascending=False)

    chosen_s = []
    s_cost = 0.0
    for _, r in df_s.iterrows():
        if len(chosen_s) >= max_supps:
            break
        if s_cost + r["typical_price"] > supp_budget:
            continue
        chosen_s.append(r)
        s_cost += r["typical_price"]

    kit_supps = pd.DataFrame(chosen_s)

    # ensure at least one collagen
    if not any(kit_supps["family"] == "Collagen"):
        extra = df_s[df_s["family"] == "Collagen"].iloc[0]
        if s_cost + extra["typical_price"] <= supp_budget and extra["name"] not in kit_supps["name"].values:
            kit_supps = pd.concat([kit_supps, pd.DataFrame([extra])], ignore_index=True)
            s_cost += extra["typical_price"]
            if len(kit_supps) > max_supps:
                kit_supps = kit_supps.sort_values("total_score", ascending=False).head(max_supps)

    # for LED Mask, try to also include HA caps
    if device_name == "LED Mask":
        has_ha = any(kit_supps["family"] == "HA")
        if not has_ha:
            ha_cand = df_s[df_s["family"] == "HA"]
            if not ha_cand.empty:
                ha_row = ha_cand.iloc[0]
                if s_cost + ha_row["typical_price"] <= supp_budget and ha_row["name"] not in kit_supps["name"].values:
                    kit_supps = pd.concat([kit_supps, pd.DataFrame([ha_row])], ignore_index=True)
                    s_cost += ha_row["typical_price"]
                    if len(kit_supps) > max_supps:
                        tmp = kit_supps.merge(df_s[["name","total_score"]], on="name", how="left")
                        mask_drop = ~tmp["family"].isin(["Collagen","HA"])
                        if mask_drop.any():
                            to_drop = tmp[mask_drop].sort_values("total_score").iloc[0]["name"]
                            drop_row = kit_supps[kit_supps["name"] == to_drop].iloc[0]
                            kit_supps = kit_supps[kit_supps["name"] != to_drop]
                            s_cost -= drop_row["typical_price"]

    # ------------------ PRICING & P&L ------------------
    bundle_topical_retail = float(kit_topicals["typical_price"].sum())
    bundle_supp_retail    = float(kit_supps["typical_price"].sum())
    suggested_price = base_price + 1.3 * (bundle_topical_retail + bundle_supp_retail)

    your_topical_cogs = bundle_topical_retail * 0.4
    your_supp_cogs    = bundle_supp_retail * 0.4
    device_cogs       = base_price * 0.35
    total_cogs        = your_topical_cogs + your_supp_cogs + device_cogs

    amz_fee = suggested_price * 0.15 + 1.80
    net_profit = suggested_price - total_cogs - amz_fee
    margin = net_profit / suggested_price * 100

    print("\n" + "="*110)
    print(f"🏥 VELORA CLINICAL PROTOCOL v2.3 → {device_name}")
    print("="*110)

    print("\nTOPICAL LAYER:")
    print(kit_topicals[["name","family","strength","typical_price","irritation_risk"]].to_string(index=False))

    print("\nINGESTIBLE LAYER:")
    print(kit_supps[["name","family","focus","typical_price","evidence_score"]].to_string(index=False))

    print("\nCOST & PRICING MODEL:")
    print(f"  • Topical Retail Stack:   ${bundle_topical_retail:.2f}")
    print(f"  • Supplement Retail Stack:${bundle_supp_retail:.2f}")
    print(f"  • Device Base Price:      ${base_price:.2f}")
    print(f"  → Suggested Bundle Price: ${suggested_price:.2f}")
    print(f"  → Estimated COGS:         ${total_cogs:.2f}")
    print(f"  → Amazon Fee (15%+1.80):  ${amz_fee:.2f}")
    print(f"  → Net Profit / Bundle:    ${net_profit:.2f}  ({margin:.1f}% margin)")
    print("="*110)

    return kit_topicals, kit_supps, suggested_price, margin

# Run v2.3 for all devices
protocols_v2_3 = {}
for d in devices:
    protocols_v2_3[d] = build_protocol_constrained_v3(
        d,
        max_topicals=4,
        max_supps=2,
        topical_budget=85.0,
        supp_budget=70.0
    )


🏥 VELORA CLINICAL PROTOCOL v2.3 → LED Mask

TOPICAL LAYER:
                               name      family strength  typical_price  irritation_risk
      2% Tranexamic Acid + Licorice Brightening      Med           23.0                1
   1.5% Hyaluronic Acid (3 weights)    Hydrator      Med           18.0                1
Ceramide Cream (3 ceramides + chol)     Barrier      Low           24.0                1
          5% Glycerin + NMF Complex    Hydrator      Low           16.0                1

INGESTIBLE LAYER:
                                    name      family                focus  typical_price  evidence_score
Hydrolyzed Collagen Peptides + Vitamin C    Collagen Elasticity/Hydration           39.0               5
           Vitamin C + Zinc Skin Complex Antioxidant      Healing/Barrier           24.0               3

COST & PRICING MODEL:
  • Topical Retail Stack:   $81.00
  • Supplement Retail Stack:$63.00
  • Device Base Price:      $199.00
  → Suggested Bundle Price: $386

In [6]:
# ==================================================================================
# 📦 CELL 2: REAL-WORLD PRODUCT SOURCING & BUNDLE OPTIMIZATION
# ==================================================================================
# Uses actual Amazon/retail pricing with shipping baked in.
# Optimizes bundle pricing assuming flat "free shipping" model.
# ==================================================================================

# Real sourced products with total landed cost (price + shipping)
real_products = {
    "10% Tranexamic Acid Serum": {
        "price": 24.56,
        "shipping": 6.99,
        "total_cost": 24.56 + 6.99,
        "family": "Brightening",
        "strength": "High",
        "size": "120ml (2x60ml)",
        "irritation_risk": 1
    },
    "Organic HA + Vitamin C Serum": {
        "price": 35.64,
        "shipping": 0.00,
        "total_cost": 35.64,
        "family": "Hydrator",
        "strength": "Med",
        "size": "1 oz",
        "irritation_risk": 1
    },
    "Cetaphil Ceramide Serum": {
        "price": 20.46,
        "shipping": 6.99,
        "total_cost": 20.46 + 6.99,
        "family": "Barrier",
        "strength": "Low",
        "size": "unknown",
        "irritation_risk": 1
    },
    "Rodial Soft Focus Glow Booster": {
        "price": 94.77,
        "shipping": 0.00,
        "total_cost": 94.77,
        "family": "Hydrator/Glow",
        "strength": "High",
        "size": "1.05 fl oz",
        "irritation_risk": 1
    },
    "Fuel Multi Collagen Protein": {
        "price": 47.96,
        "shipping": 0.00,
        "total_cost": 47.96,
        "family": "Collagen",
        "strength": "N/A",
        "size": "powder",
        "evidence_score": 5,
        "focus": "Elasticity/Hydration"
    },
    "NAD+ Immunity Capsules (D3 K2 C Zinc Elderberry Quercetin)": {
        "price": 24.51,
        "shipping": 6.99,
        "total_cost": 24.51 + 6.99,
        "family": "Antioxidant/Barrier",
        "strength": "N/A",
        "size": "capsules",
        "evidence_score": 3,
        "focus": "Immunity/Barrier"
    },
    "Polyglutamic Acid Serum 10%": {
        "price": 15.09,
        "shipping": 6.99,
        "total_cost": 15.09 + 6.99,
        "family": "Hydrator",
        "strength": "High",
        "size": "unknown",
        "irritation_risk": 1
    },
    "IUNIK Beta Glucan Daily Moisture Cream": {
        "price": 36.26,
        "shipping": 6.99,
        "total_cost": 36.26 + 6.99,
        "family": "Soothing/Barrier",
        "strength": "Low",
        "size": "unknown",
        "irritation_risk": 1
    },
    "NAD+ Resveratrol Liposomal NR 900mg": {
        "price": 54.98,
        "shipping": 0.00,
        "total_cost": 54.98,
        "family": "NAD+/Barrier",
        "strength": "High",
        "size": "capsules",
        "evidence_score": 3,
        "focus": "Cellular Energy/NAD"
    }
}

# Device base prices (unchanged)
device_base_prices_real = {
    "LED Mask": 199.00,
    "Microcurrent Massager": 119.99,
    "Knee Massager": 119.99,
    "High Frequency Wand": 189.99,
    "Electric Gua Sha": 119.99
}

# Define custom bundles based on your specifications
real_bundles = {
    "LED Mask": {
        "topical": [
            "10% Tranexamic Acid Serum",
            "Organic HA + Vitamin C Serum",
            "Cetaphil Ceramide Serum",
            "Rodial Soft Focus Glow Booster"
        ],
        "supplement": [
            "Fuel Multi Collagen Protein",
            "NAD+ Immunity Capsules (D3 K2 C Zinc Elderberry Quercetin)"
        ]
    },
    "Microcurrent Massager": {
        "topical": [
            "Polyglutamic Acid Serum 10%",
            "Cetaphil Ceramide Serum",
            "Organic HA + Vitamin C Serum",
            "Rodial Soft Focus Glow Booster"
        ],
        "supplement": [
            "Fuel Multi Collagen Protein",
            "NAD+ Immunity Capsules (D3 K2 C Zinc Elderberry Quercetin)"
        ]
    },
    "Knee Massager": {
        "topical": [
            "IUNIK Beta Glucan Daily Moisture Cream",
            "Organic HA + Vitamin C Serum",
            "Cetaphil Ceramide Serum",
            "Rodial Soft Focus Glow Booster"
        ],
        "supplement": [
            "Fuel Multi Collagen Protein",
            "NAD+ Resveratrol Liposomal NR 900mg"
        ]
    },
    "High Frequency Wand": {
        "topical": [
            "Organic HA + Vitamin C Serum",
            "IUNIK Beta Glucan Daily Moisture Cream",
            "Cetaphil Ceramide Serum",
            "Rodial Soft Focus Glow Booster"
        ],
        "supplement": [
            "Fuel Multi Collagen Protein",
            "NAD+ Resveratrol Liposomal NR 900mg"
        ]
    },
    "Electric Gua Sha": {
        "topical": [
            "IUNIK Beta Glucan Daily Moisture Cream",
            "Cetaphil Ceramide Serum",
            "Organic HA + Vitamin C Serum",
            "Rodial Soft Focus Glow Booster"
        ],
        "supplement": [
            "Fuel Multi Collagen Protein",
            "NAD+ Resveratrol Liposomal NR 900mg"
        ]
    }
}

def calculate_real_bundle_pricing(device_name, bundle_spec, real_products, device_base_prices_real):
    """
    Calculate bundle pricing with real product data.
    Shipping is baked into product cost; bundle price assumes 'free shipping'.
    """
    base_price = device_base_prices_real[device_name]
    
    # Sum topical costs
    topical_cost = sum(real_products[name]["total_cost"] for name in bundle_spec["topical"])
    topical_retail = topical_cost  # For display, same as cost
    
    # Sum supplement costs
    supp_cost = sum(real_products[name]["total_cost"] for name in bundle_spec["supplement"])
    supp_retail = supp_cost
    
    # Bundle pricing: cost + markup
    # Calculate cost baseline
    total_product_cost = topical_cost + supp_cost
    device_cogs = base_price * 0.35
    total_bundle_cogs = total_product_cost + device_cogs
    
    # Suggested retail (1.8x multiplier on product costs + device base)
    suggested_bundle_price = base_price + 1.8 * (topical_cost + supp_cost)
    
    # Amazon fees & profit
    amz_fee = suggested_bundle_price * 0.15 + 1.80
    net_profit = suggested_bundle_price - total_bundle_cogs - amz_fee
    margin_pct = (net_profit / suggested_bundle_price * 100) if suggested_bundle_price > 0 else 0
    
    return {
        "device": device_name,
        "base_device_price": base_price,
        "topical_bundle_cost": topical_cost,
        "topical_items": bundle_spec["topical"],
        "supp_bundle_cost": supp_cost,
        "supp_items": bundle_spec["supplement"],
        "total_product_cost": total_product_cost,
        "device_cogs": device_cogs,
        "total_cogs": total_bundle_cogs,
        "suggested_bundle_price": suggested_bundle_price,
        "amz_fee": amz_fee,
        "net_profit": net_profit,
        "margin_pct": margin_pct
    }

# ==================================================================================
# REAL BUNDLE CALCULATIONS
# ==================================================================================

print("\n" + "="*130)
print("💎 VELORA v2.3 REAL-WORLD BUNDLE OPTIMIZATION (CELL 2)")
print("="*130)

real_bundle_results = {}
for device_name, bundle_spec in real_bundles.items():
    result = calculate_real_bundle_pricing(device_name, bundle_spec, real_products, device_base_prices_real)
    real_bundle_results[device_name] = result
    
    print(f"\n🏥 {device_name.upper()}")
    print("-"*130)
    
    print("\nTOPICAL LAYER (Real Sourced):")
    for item in bundle_spec["topical"]:
        prod = real_products[item]
        print(f"  • {item}")
        print(f"    Price: ${prod['price']:.2f} + Ship: ${prod['shipping']:.2f} = ${prod['total_cost']:.2f}")
    
    print(f"\n  TOPICAL SUBTOTAL: ${result['topical_bundle_cost']:.2f}")
    
    print("\nINGESTIBLE LAYER (Real Sourced):")
    for item in bundle_spec["supplement"]:
        prod = real_products[item]
        print(f"  • {item}")
        print(f"    Price: ${prod['price']:.2f} + Ship: ${prod['shipping']:.2f} = ${prod['total_cost']:.2f}")
    
    print(f"\n  SUPPLEMENT SUBTOTAL: ${result['supp_bundle_cost']:.2f}")
    
    print("\nPRICING & P&L:")
    print(f"  Device Base Price:              ${result['base_device_price']:.2f}")
    print(f"  Product Total Cost:             ${result['total_product_cost']:.2f}")
    print(f"  Device COGS (35% of base):      ${result['device_cogs']:.2f}")
    print(f"  → Total Bundle COGS:            ${result['total_cogs']:.2f}")
    print(f"\n  → Suggested Bundle Price:       ${result['suggested_bundle_price']:.2f}")
    print(f"  → Amazon Fee (15% + $1.80):     ${result['amz_fee']:.2f}")
    print(f"  → Net Profit per Bundle:        ${result['net_profit']:.2f}")
    print(f"  → Profit Margin:                {result['margin_pct']:.1f}%")
    print("="*130)

# Summary table
print("\n" + "="*130)
print("📊 REAL BUNDLE SUMMARY")
print("="*130)

summary_df = pd.DataFrame([
    {
        "Device": result["device"],
        "Topical Cost": f"${result['topical_bundle_cost']:.2f}",
        "Supp Cost": f"${result['supp_bundle_cost']:.2f}",
        "Total Product Cost": f"${result['total_product_cost']:.2f}",
        "Suggested Price": f"${result['suggested_bundle_price']:.2f}",
        "COGS": f"${result['total_cogs']:.2f}",
        "Net Profit": f"${result['net_profit']:.2f}",
        "Margin %": f"{result['margin_pct']:.1f}%"
    }
    for result in real_bundle_results.values()
])

print(summary_df.to_string(index=False))
print("="*130)


💎 VELORA v2.3 REAL-WORLD BUNDLE OPTIMIZATION (CELL 2)

🏥 LED MASK
----------------------------------------------------------------------------------------------------------------------------------

TOPICAL LAYER (Real Sourced):
  • 10% Tranexamic Acid Serum
    Price: $24.56 + Ship: $6.99 = $31.55
  • Organic HA + Vitamin C Serum
    Price: $35.64 + Ship: $0.00 = $35.64
  • Cetaphil Ceramide Serum
    Price: $20.46 + Ship: $6.99 = $27.45
  • Rodial Soft Focus Glow Booster
    Price: $94.77 + Ship: $0.00 = $94.77

  TOPICAL SUBTOTAL: $189.41

INGESTIBLE LAYER (Real Sourced):
  • Fuel Multi Collagen Protein
    Price: $47.96 + Ship: $0.00 = $47.96
  • NAD+ Immunity Capsules (D3 K2 C Zinc Elderberry Quercetin)
    Price: $24.51 + Ship: $6.99 = $31.50

  SUPPLEMENT SUBTOTAL: $79.46

PRICING & P&L:
  Device Base Price:              $199.00
  Product Total Cost:             $268.87
  Device COGS (35% of base):      $69.65
  → Total Bundle COGS:            $338.52

  → Suggested Bundle Price